In [4]:
import sys; sys.path.insert(0, "../src")
import pandas as pd, numpy as np
from elo import add_elo
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

df = pd.read_csv("../data/processed/laliga_matches.csv", parse_dates=["Date"], dtype={"Season": str})
df = add_elo(df)
df["y"] = df["FTR"].map({"H": 0, "D": 1, "A": 2})

train = df[df["Season"].isin(["1617","1718","1819","1920","2021","2122","2223"])]
test  = df[df["Season"].isin(["2324","2425","2526"])]

X_tr, y_tr = train[["elo_diff"]], train["y"]
X_te, y_te = test[["elo_diff"]], test["y"]

modelo = LogisticRegression(max_iter=1000)
modelo.fit(X_tr, y_tr)
P = modelo.predict_proba(X_te)

print("Log-loss Elo + logística:", round(log_loss(y_te, P, labels=[0,1,2]), 4))

# Referencias sobre el MISMO test, sin fuga
freq = np.bincount(y_tr) / len(y_tr)
P_base = np.tile(freq, (len(y_te), 1))
print("Log-loss línea base (freq. de train):", round(log_loss(y_te, P_base, labels=[0,1,2]), 4))

m = test.dropna(subset=["AvgCH","AvgCD","AvgCA"])
inv = 1 / m[["AvgCH","AvgCD","AvgCA"]].values
P_mkt = inv / inv.sum(axis=1, keepdims=True)
print("Log-loss mercado (mismo test):", round(log_loss(m["y"], P_mkt, labels=[0,1,2]), 4))

Log-loss Elo + logística: 0.9746
Log-loss línea base (freq. de train): 1.065
Log-loss mercado (mismo test): 0.9537


## Baseline: Elo + regresión logística

**Diseño (sin fuga de datos):**
- El rating Elo de cada equipo se calcula únicamente con partidos anteriores
  (se guarda el rating antes de actualizarlo con el resultado).
- Entrenamiento: temporadas 2016/17 a 2022/23. Test: 2023/24 a 2025/26
  (1.140 partidos).
- Parámetros de Elo fijados a mano (K = 20, ventaja local = 60,
  regresión a la media = 25 % por temporada), sin optimizar.

**Resultados (log-loss, menor es mejor):**

| Modelo | Log-loss |
|---|---|
| Línea base (frecuencias de train) | 1,0650 |
| Elo + logística | 0,9746 |
| Mercado (AvgC) | 0,9537 |

**Lectura:** con una sola variable, el Elo recupera ≈ 81 % de la distancia
entre la línea base y el mercado. La brecha restante es 0,021. Falta
cuantificar su incertidumbre y validar con varios cortes temporales.

In [5]:
rng = np.random.default_rng(42)

# Pérdida por partido: -ln(prob. asignada al resultado real)
idx = np.arange(len(m))
y_m = m["y"].values
P_elo_m = modelo.predict_proba(m[["elo_diff"]])

loss_elo = -np.log(P_elo_m[idx, y_m])
loss_mkt = -np.log(P_mkt[idx, y_m])
loss_base = -np.log(P_base[:len(m)][idx, y_m])

B = 10000
diffs_mkt, diffs_base = [], []
for _ in range(B):
    s = rng.integers(0, len(m), len(m))        # remuestra con reemplazamiento
    diffs_mkt.append(loss_elo[s].mean() - loss_mkt[s].mean())
    diffs_base.append(loss_elo[s].mean() - loss_base[s].mean())

def resumen(d, nombre):
    lo, hi = np.percentile(d, [2.5, 97.5])
    print(f"{nombre}: media={np.mean(d):+.4f}  IC95%=[{lo:+.4f}, {hi:+.4f}]")

resumen(diffs_mkt, "Elo - mercado")
resumen(diffs_base, "Elo - línea base")

Elo - mercado: media=+0.0209  IC95%=[+0.0100, +0.0321]
Elo - línea base: media=-0.0905  IC95%=[-0.1127, -0.0672]


### Incertidumbre: bootstrap pareado del log-loss

Remuestreamos con reemplazamiento los 1.140 partidos del test (10.000 veces),
usando los mismos partidos para todos los modelos, y calculamos la diferencia
de log-loss.

| Comparación | Diferencia media | IC 95 % |
|---|---|---|
| Elo − mercado | +0,0209 | [+0,0100, +0,0321] |
| Elo − línea base | −0,0905 | [−0,1127, −0,0672] |

**Lectura:** el Elo mejora la línea base de forma clara. El mercado sigue
siendo mejor, con una brecha pequeña (≈ 0,02) cuyo intervalo no incluye 0.

**Limitaciones:** el bootstrap trata los partidos como independientes
(los equipos se repiten y su fuerza persiste), por lo que el intervalo puede
ser algo optimista; y refleja un único corte temporal.

### Bootstrap por bloques de jornada

En primer lugar definimos los bloques

In [6]:
m = m.copy()
m["bloque"] = m["Date"].dt.to_period("W").astype(str)
print(m["bloque"].nunique(), "bloques")
print(m.groupby("bloque").size().describe())

110 bloques
count    110.000000
mean      10.363636
std        3.233220
min        1.000000
25%        9.000000
50%       10.000000
75%       11.000000
max       21.000000
dtype: float64


In [7]:
rng = np.random.default_rng(42)

# Pérdidas por partido (ya calculadas arriba: loss_elo, loss_mkt, loss_base)
df_l = pd.DataFrame({
    "bloque": m["bloque"].values,
    "elo": loss_elo, "mkt": loss_mkt, "base": loss_base,
})

# Sumas y tamaños por bloque
g = df_l.groupby("bloque").agg(
    n=("elo", "size"), elo=("elo", "sum"), mkt=("mkt", "sum"), base=("base", "sum")
)
n_b = len(g)

B = 10000
d_mkt, d_base = [], []
for _ in range(B):
    s = rng.integers(0, n_b, n_b)                  # remuestra BLOQUES
    tot = g.iloc[s].sum()
    d_mkt.append((tot["elo"] - tot["mkt"]) / tot["n"])
    d_base.append((tot["elo"] - tot["base"]) / tot["n"])

resumen(d_mkt, "Elo - mercado (bloques)")
resumen(d_base, "Elo - línea base (bloques)")

Elo - mercado (bloques): media=+0.0209  IC95%=[+0.0101, +0.0319]
Elo - línea base (bloques): media=-0.0905  IC95%=[-0.1127, -0.0680]


### Incertidumbre: bootstrap pareado por bloques

Para respetar la dependencia entre partidos de una misma jornada, repetimos
el bootstrap remuestreando **semanas completas** (110 bloques, ≈ 10 partidos
cada uno) en lugar de partidos sueltos. 10.000 remuestras, mismos partidos
para todos los modelos.

| Comparación | Bootstrap simple | Bootstrap por bloques |
|---|---|---|
| Elo − mercado | [+0,0100, +0,0321] | [+0,0101, +0,0319] |
| Elo − línea base | [−0,1127, −0,0672] | [−0,1127, −0,0680] |

**Lectura:** los intervalos apenas cambian, así que la conclusión es robusta
a la dependencia dentro de la jornada: el Elo mejora claramente la línea base
y el mercado sigue siendo algo mejor (brecha ≈ 0,02, intervalo sin incluir 0).

**Limitaciones:** los bloques semanales no recogen la dependencia entre
jornadas (rachas de un mismo equipo), y el intervalo refleja un único corte
train/test.